### Просмотр вперёд (Lookahead) в регулярных выражениях Python

Разберём механизм, который часто называют «контекстным фильтром» регулярных выражений. Это просмотр вперёд, или lookahead. 

Его суть проста: мы проверяем, что идёт после текущего положения курсора, но не включаем эту проверку в результат совпадения и не сдвигаем курсор.

В терминологии движка `re` такие конструкции называются утверждениями нулевой ширины (zero-width assertions). Они не потребляют символы, а только отвечают на вопрос: «верно ли, что дальше идёт (или не идёт) указанный шаблон?».

####  Синтаксис в Python

В стандартном модуле `re` существуют две формы:

| Тип | Синтаксис | Логика |
| --- | --- | --- |
| Положительный | `(?=...)` | Совпадение возможно, только если дальше идёт шаблон `...` |
| Отрицательный | `(?!...)` | Совпадение возможно, только если дальше **не** идёт шаблон `...` |


`(?=(...))` ищет все возможные вхождения, даже с учетом пересечений

Пример.
Допустим, нам нужно извлечь только суммы, после которых стоит валюта ₽, но сам символ валюты в результат попадать не должен.

In [4]:
import re
text = "аренда 5000₽, залог 10000, коммуналка 3000₽"

# Без lookahead: захватит и цифры, и символ
res = re.findall(r'\d+₽', text) # ['5000₽', '3000₽']
print(res)

# С положительным просмотром:
pattern = r'\d+(?=₽)'
res = re.findall(pattern, text)  # ['5000', '3000']
print(res)

['5000₽', '3000₽']
['5000', '3000']


Что произошло внутри движка:
\d+ находит 5000.
Курсор остаётся перед ₽.
(?=₽) проверяет: «да, дальше символ ₽». Условие выполнено.
Совпадение фиксируется как 5000, курсор возвращается на начало ₽ и продолжает поиск со следующего символа.

Задача: найти все теги `<img>`, у которых нет атрибута loading="lazy".

In [5]:
html = '<img src="1.jpg" loading="lazy"> <img src="2.png"> <div>...</div>'
# Простой вариант (демонстрация логики):
re.findall(r'<img(?![^>]*loading="lazy")[^>]*>', html)

['<img src="2.png">']

Здесь (?!...) проверяет, что внутри открывающего тега не встретится указанная строка. Если встретится, совпадение отвергается целиком.

Важные нюансы и подводные камни:

- Нулевая ширина ≠ отсутствие накладных расходов. Утверждения всё равно вычисляются. Вложенные квантификаторы внутри (?=...) могут вызвать катастрофический бэктрекинг. Пример-антиспаттерн: (?=.*?.*?) на длинных строках. Пишите проверку максимально конкретно.
- Порядок имеет значение. (?=...) проверяет условие до того, как основное выражение продолжит работу. Поэтому (?=.)\w+ и \w+(?=.) дадут разные результаты на граничных позициях.
- Python re vs regex. В стандартном модуле re просмотр вперёд поддерживает произвольную длину шаблона, а вот просмотр назад ((?<=...)) требует фиксированной ширины. Если вам нужна симметрия, смотрите в сторону стороннего пакета regex, но в 90% задач (?=...) и (?!...) покрывают потребности без лишних зависимостей.

Lookahead — это не про извлечение текста, а про условную валидацию позиции. Он идеально подходит для:
- проверки формата (пароли, телефоны, логи),
- фильтрации токенов по правому контексту,
- безопасного парсинга, когда нельзя «съедать» разделители.